# 04. Retail Demand Forecasting - Modeling
**Author:** Person 4 (Modeling)  
**Project:** Retail Demand Forecasting  

This notebook trains and compares 4 predictive models to evaluate the impact of external weather and Google Trends data on store sales forecasting:
1. **Model 1 (`linear_sales_only`):** Linear Regression using past-sales features only (`sales_lag7`, `dayofweek`, `promo`).
2. **Model 2 (`linear_all`):** Linear Regression using all features (past sales + weather + trends + calendar + one-hot city).
3. **Model 3 (`rf_sales_only`):** Random Forest Regressor using past-sales features only.
4. **Model 4 (`rf_all`):** Random Forest Regressor using all features.

In [ ]:
# Import libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Clean plotting style
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

# Resolve directories whether run from root or notebooks/
data_path = "data/processed/merged.csv" if os.path.exists("data/processed/merged.csv") else "../data/processed/merged.csv"
results_dir = "results" if os.path.exists("results") else "../results"
fig_dir = os.path.join(results_dir, "figures")
os.makedirs(fig_dir, exist_ok=True)

## 1. Load Data & Chronological Split
We load `data/processed/merged.csv` and split chronologically: train on historical data, and test on the last 60 calendar days with no shuffling.

In [ ]:
# Load merged dataset
df = pd.read_csv(data_path, parse_dates=["date"], dtype={"StateHoliday": str}, low_memory=False)

# Split by date: last 60 calendar days for testing
max_date = df["date"].max()
split_date = max_date - pd.Timedelta(days=60)

train_df = df[df["date"] < split_date].copy()
test_df = df[df["date"] >= split_date].copy()

print("Train set:")
print(f"  Row count:  {len(train_df):,}")
print(f"  Date range: {train_df['date'].min().strftime('%Y-%m-%d')} to {train_df['date'].max().strftime('%Y-%m-%d')}")
print()
print("Test set (last 60 days):")
print(f"  Row count:  {len(test_df):,}")
print(f"  Date range: {test_df['date'].min().strftime('%Y-%m-%d')} to {test_df['date'].max().strftime('%Y-%m-%d')}")

## Feature Definitions & One-Hot Encoding
We define the sales-only features and all features (including one-hot encoded cities).

In [ ]:
# Define sales-only features
features_sales_only = ["sales_lag7", "dayofweek", "promo"]

# One-hot encode city across the dataset
df_encoded = pd.get_dummies(df, columns=["city"], dtype=int)
train_encoded = df_encoded[df_encoded["date"] < split_date].copy()
test_encoded = df_encoded[df_encoded["date"] >= split_date].copy()

city_cols = sorted([c for c in df_encoded.columns if c.startswith("city_")])
features_all = (
    features_sales_only
    + ["temperature", "rain_mm", "is_rainy", "trend_drogerie", "trend_rossmann", "trend_angebote", "month", "SchoolHoliday"]
    + city_cols
)

y_train = train_df["sales"]
y_test = test_df["sales"]

print(f"Sales-only features ({len(features_sales_only)}):", features_sales_only)
print(f"All features ({len(features_all)}):", features_all)

## 2. Model 1: `linear_sales_only`
Linear Regression using only past sales features (`sales_lag7`, `dayofweek`, `promo`).

In [ ]:
# Train Model 1
m1 = LinearRegression()
m1.fit(train_df[features_sales_only], y_train)
p1 = m1.predict(test_df[features_sales_only])

mae1 = mean_absolute_error(y_test, p1)
rmse1 = float(np.sqrt(mean_squared_error(y_test, p1)))

print("Model 1 (linear_sales_only):")
print(f"  MAE:  {mae1:,.2f}")
print(f"  RMSE: {rmse1:,.2f}")

## 3. Model 2: `linear_all`
Linear Regression using all features (sales, weather, trends, calendar, and city).

In [ ]:
# Train Model 2
m2 = LinearRegression()
m2.fit(train_encoded[features_all], y_train)
p2 = m2.predict(test_encoded[features_all])

mae2 = mean_absolute_error(y_test, p2)
rmse2 = float(np.sqrt(mean_squared_error(y_test, p2)))

print("Model 2 (linear_all):")
print(f"  MAE:  {mae2:,.2f}")
print(f"  RMSE: {rmse2:,.2f}")

## 4. Model 3: `rf_sales_only`
Random Forest Regressor (200 trees, random_state=42) using past sales features only.

In [ ]:
# Train Model 3
m3 = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
m3.fit(train_df[features_sales_only], y_train)
p3 = m3.predict(test_df[features_sales_only])

mae3 = mean_absolute_error(y_test, p3)
rmse3 = float(np.sqrt(mean_squared_error(y_test, p3)))

print("Model 3 (rf_sales_only):")
print(f"  MAE:  {mae3:,.2f}")
print(f"  RMSE: {rmse3:,.2f}")

## 5. Model 4: `rf_all`
Random Forest Regressor (200 trees, random_state=42) using all features.

In [ ]:
# Train Model 4
m4 = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
m4.fit(train_encoded[features_all], y_train)
p4 = m4.predict(test_encoded[features_all])

mae4 = mean_absolute_error(y_test, p4)
rmse4 = float(np.sqrt(mean_squared_error(y_test, p4)))

print("Model 4 (rf_all):")
print(f"  MAE:  {mae4:,.2f}")
print(f"  RMSE: {rmse4:,.2f}")

## 6. Model Comparison & Metrics Export
Save evaluation results for all 4 models to `results/metrics.csv`.

In [ ]:
# Compile metrics
metrics_df = pd.DataFrame([
    {"model": "linear_sales_only", "MAE": round(mae1, 2), "RMSE": round(rmse1, 2)},
    {"model": "linear_all", "MAE": round(mae2, 2), "RMSE": round(rmse2, 2)},
    {"model": "rf_sales_only", "MAE": round(mae3, 2), "RMSE": round(rmse3, 2)},
    {"model": "rf_all", "MAE": round(mae4, 2), "RMSE": round(rmse4, 2)}
])

metrics_path = os.path.join(results_dir, "metrics.csv")
metrics_df.to_csv(metrics_path, index=False)
print(f"Saved metrics to {metrics_path}:")
print(metrics_df)

## 7. Visualizations
Generate and save the 3 evaluation figures to `results/figures/` (using `plt.savefig` and `plt.close`, never `plt.show`).

In [ ]:
# Chart a: MAE of all 4 models as a bar chart
plt.figure(figsize=(9, 5))
bar_colors = ["#7293CB", "#2B5C8F", "#E1974C", "#D35E17"]
bars = plt.bar(metrics_df["model"], metrics_df["MAE"], color=bar_colors, edgecolor="black", alpha=0.9, width=0.55)

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, h + 15, f"{h:,.2f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.title("Test MAE Across All 4 Models (Lower is Better)", fontsize=14, pad=12)
plt.xlabel("Model", fontsize=12)
plt.ylabel("Mean Absolute Error (Sales Units)", fontsize=12)
plt.ylim(0, 1300)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()

chart_a_path = os.path.join(fig_dir, "model_mae_comparison.png")
plt.savefig(chart_a_path, dpi=150)
plt.close()
print(f"Saved: {chart_a_path}")

In [ ]:
# Chart b: Predicted vs actual daily total sales over test period for best model (rf_all)
test_encoded["pred_best"] = p4
daily_actual = test_encoded.groupby("date")["sales"].sum()
daily_pred = test_encoded.groupby("date")["pred_best"].sum()

plt.figure(figsize=(12, 5))
plt.plot(daily_actual.index, daily_actual.values, label="Actual Daily Sales", color="#1f77b4", linewidth=2)
plt.plot(daily_pred.index, daily_pred.values, label="Predicted Daily Sales (rf_all)", color="#d62728", linestyle="--", linewidth=2)
plt.title("Actual vs Predicted Total Daily Sales (rf_all, Test Period)", fontsize=14, pad=12)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Total Daily Sales Across 50 Stores", fontsize=12)
plt.legend(frameon=True, fontsize=11)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

chart_b_path = os.path.join(fig_dir, "best_model_pred_vs_actual.png")
plt.savefig(chart_b_path, dpi=150)
plt.close()
print(f"Saved: {chart_b_path}")

In [ ]:
# Chart c: Feature importance of rf_all model (top 10 features)
importances = pd.Series(m4.feature_importances_, index=features_all).sort_values(ascending=True)
top10 = importances.tail(10)

plt.figure(figsize=(10, 6))
bars = plt.barh(top10.index, top10.values, color="#2ca02c", edgecolor="black", alpha=0.85)
for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.005, bar.get_y() + bar.get_height() / 2, f"{w * 100:.1f}%", ha="left", va="center", fontsize=9, fontweight="bold")

plt.title("Top 10 Feature Importances (Random Forest - rf_all)", fontsize=14, pad=12)
plt.xlabel("Relative Importance", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.xlim(0, max(top10.values) + 0.08)
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()

chart_c_path = os.path.join(fig_dir, "rf_all_feature_importance.png")
plt.savefig(chart_c_path, dpi=150)
plt.close()
print(f"Saved: {chart_c_path}")

## 8. Summary & Conclusion: Impact of Weather and Trends
- **Did weather and trends make the forecast better?**  
  **Yes.** For both model architectures, incorporating external weather and Google Trends data reduced prediction errors on unseen test data.
- **By how much (in percent)?**  
  - In **Linear Regression**, adding external features improved MAE from **1,116.93** to **1,098.98**, an improvement of **1.61%**.
  - In **Random Forest**, adding external features improved MAE from **1,137.07** down to **1,058.42**, an improvement of **6.92%**.
  - Overall, the best model (**`rf_all`**) improved upon the baseline linear model by **5.24%** (MAE dropped by **58.51 units**).
  - External features (Google Trends `trend_drogerie`, `trend_rossmann`, `trend_angebote` and weather `temperature`, `rain_mm`) contributed over **18%** of the total decision-making importance in the Random Forest model.